# EXP-002 — CodeBERT fine-tuning trên Kaggle GPU

**Setup 1 lần trên Kaggle:** Settings → Accelerator = **GPU**, Internet = **ON**.

**Sửa đúng 1 chỗ:** cell 2 — dòng `REPO = "<user>/<repo>"`.
Sau đó chạy cell theo thứ tự. **Đừng bấm Run All** — cell 7 (train) mất ~8h.

### Vì sao notebook này khác bản cũ
1. Bản cũ dùng `!git clone $url` — IPython **không** nội suy biến Python trong `!`,
   `$url` bị bash expand thành chuỗi rỗng → `git clone` chết ngay cell 2. Bản này dùng `subprocess`.
2. Artifact flush **mỗi epoch** (DEC-018) + `EVAL_COMPLETE.json`, nên session chết vẫn còn kết quả.
3. `--time-budget-s` dừng train graceful trước khi Kaggle kill phiên 12h, kịp chạy evaluate.
4. Nếu session chết giữa chừng → chạy lại cell 7 với `MODE = "resume"` để train tiếp, không chạy lại từ đầu.
5. **Evaluate chạy ngay trên Kaggle** (cell 8) và chỉ tải metrics + predictions về
   (vài MB) — không tải checkpoint 1,4GB về chạy lại local.


In [ ]:
import json
import os
import subprocess
import sys

# Cách 1 (khuyên dùng): Kaggle Add-ons → Secrets → thêm GITHUB_TOKEN, HF_TOKEN, attach vào notebook
# Cách 2: dán trực tiếp dưới đây (đỪNG share notebook này)
GITHUB_TOKEN = ""  # GitHub PAT — bắt buộc nếu repo PRIVATE
HF_TOKEN = ""  # HuggingFace token — tải dataset/checkpoint ổn định
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as e:
    print("dùng token dán tay:", e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN  # huggingface_hub/datasets tự đọc biến này
print("GITHUB_TOKEN:", "OK" if GITHUB_TOKEN else "TRỐNG", "| HF_TOKEN:", "OK" if HF_TOKEN else "TRỐNG")


In [ ]:
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sửa REPO trước khi chạy cell tiếp theo"

import subprocess

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lệnh thất bại (rc={r.returncode}): {cmd}")
    return r.stdout

def py(code, **kw):
    """Chạy snippet Python một dòng (không dùng newline trong code)."""
    return sh(f'{sys.executable} -c {json.dumps(code)}', **kw)

# Idempotent: sau khi session chết và restart notebook để resume, cell này
# chạy lại sẽ pull bản mới thay vì clone lần nữa (clone trùng sẽ fail).
if os.path.isdir("project"):
    print("project đã tồn tại → pull thay vì clone")
    sh("git -C project pull --ff-only")
else:
    url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git" if GITHUB_TOKEN else f"https://github.com/{REPO}.git"
    sh(f'git clone "{url}" project')
sh(f"git -C project remote set-url origin https://github.com/{REPO}.git")  # xóa token khỏi remote
sh("git -C project log --oneline -1")
os.chdir("project")
print("cwd =", os.getcwd())
leak = GITHUB_TOKEN and GITHUB_TOKEN in sh("git remote -v")
print("còn sót token trong remote?", "CÓ — xóa notebook!" if leak else "KHÔNG")


In [ ]:
# Cài phần còn thiếu. KHÔNG đụng torch: Kaggle đã có sẵn bản CUDA tương thích,
# pip install transformers vô tình kéo torch về bản CPU/default sẽ làm train rất chậm (hoặc chết).
import importlib

need = []
for mod, pkg in [("transformers", "transformers"), ("datasets", "datasets"),
                 ("sklearn", "scikit-learn"), ("yaml", "pyyaml"),
                 ("tqdm", "tqdm"), ("numpy", "numpy"),
                 ("tree_sitter_cpp", "tree-sitter-cpp")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print("cần cài:", need or "không")
if need:
    sh("pip install -q " + " ".join(need))

import torch, transformers, datasets, sklearn, numpy
assert torch.cuda.is_available(), "Kaggle chưa gắn GPU — vào Settings → Accelerator → GPU"
print({"torch": torch.__version__, "cuda": torch.version.cuda, "transformers": transformers.__version__,
       "datasets": datasets.__version__, "gpu": torch.cuda.get_device_name(0),
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory/1e9, 1)})


In [ ]:
# Rebuild dataset foundation POJ-104 ngay trên Kaggle.
# data/processed + data/splits bị .gitignore nên không có sẵn trong clone — phải build lại.
sh("python scripts/build_dataset.py --config configs/pairs/poj104.yaml")
py("import json; m = json.load(open('data/manifests/poj104.json')); "
   "print('leakage_passed =', m['leakage_passed']); "
   "print('pairs =', {k: v for k, v in m.get('pairs', {}).items()})")
sh("python scripts/env_report.py --output results/exp002_codebert_poj104/env.json")
sh("df -h /kaggle/working | tail -1")


In [ ]:
# Preflight: GPU/VRAM/model/config trước khi tốn 8h
sh("python -m src.experiments.exp002_codebert --config configs/exp002_codebert.yaml "
   "--mode preflight --device cuda")


In [ ]:
# Diagnostic: xác nhận model/gradient/param thật sự chạy trên GPU.
# Đây là gate của reports/010 — phải PASS trước khi train.
sh("python -m src.experiments.exp002_codebert --config configs/exp002_codebert.yaml "
   "--mode diagnostic --device cuda")


In [ ]:
# FULL TRAINING — cell này mất ~8h, đừng chạy Run All.
#
# Nếu session bị kill giữa chừng: đổi MODE = "resume" rồi chạy lại cell này.
# Nạp last_checkpoint.pt, train tiếp epoch còn lại, không mất tiến độ.
MODE = "train"  # "train" | "resume"
TIME_BUDGET_S = 9 * 3600  # dừng graceful trước giới hạn 12h của Kaggle
sh(f"python -m src.experiments.exp002_codebert --config configs/exp002_codebert.yaml "
   f"--mode {MODE} --device cuda --time-budget-s {TIME_BUDGET_S}")


In [ ]:
# EVALUATE — chạy ngay trên Kaggle (~5 phút), KHÔNG tải checkpoint về máy.
#
# Vì sao cần cell này khi `train` đã tự evaluate: để artifact
# (metrics.json, predictions/*.jsonl, EVAL_COMPLETE.json) được sinh bởi MỘT lần
# chạy có thứ tự rõ ràng, và để chạy lại được sau khi tải xuống mà không tốn giờ local.
# Chạy lại nhiều lần là an toàn: ghi đè cùng một thư mục kết quả, không nhân bản.
sh("python -m src.experiments.exp002_codebert --config configs/exp002_codebert.yaml "
   "--mode evaluate --device cuda")
OUT = "results/exp002_codebert_poj104"
sh(f"test -f {OUT}/EVAL_COMPLETE.json && echo 'EVAL COMPLETE' || echo 'EVAL CHUA XONG'")
print(sh(f"cat {OUT}/metrics.json"))


In [ ]:
# Đóng gói kết quả để tải về — CHỈ metrics + predictions (vài MB), không kèm
# checkpoint: đã evaluate xong ở cell trên, và predictions/*.jsonl đủ để tính lại
# MỌI metric/threshold/error analysis mà không cần model (reports/011 §11.13).
OUT = "results/exp002_codebert_poj104"
# Đổi sang True chỉ khi muốn giữ `best_weights_only.pt` (0,5GB, từ 1,44GB) để chạy
# lại evaluate ở máy khác. Mặc định False ⇒ tải về nhẹ nhất.
KEEP_WEIGHTS = False
if KEEP_WEIGHTS:
    py(f"from pathlib import Path; from src.experiments.exp002_codebert import export_eval_weights; export_eval_weights(Path('{OUT}'))")

sh(f"tar --exclude='*checkpoint.pt' --exclude='*.tmp' -czf exp002_results.tar.gz {OUT}")
sh("ls -lh exp002_results.tar.gz")
sh(f"ls -lh {OUT}/")
print(sh(f"cat {OUT}/metrics.json"))


### Sau khi tải `exp002_results.tar.gz` về (vài MB)
1. Giải nén **đè** lên `results/exp002_codebert_poj104/`.
2. **Không cần** chạy evaluate local — `metrics.json`, `threshold.json`,
   `confusion_matrix.json`, `model_stats.json` và `predictions/{validation,test}_scores.jsonl`
   đã có sẵn. Muốn tính lại số khác (F1 ở threshold khác, error analysis theo độ dài…)
   thì đọc `predictions/`, **không cần GPU**.
3. Nếu session Kaggle đã chết giữa chừng và bạn còn `last_checkpoint.pt` trong
   version đã save: chép vào thư mục result rồi `--mode resume`.
4. Cập nhật docs (repo yêu cầu code + docs tiến song):
   - `reports/011_codebert_results.md` — kết quả test, so sánh với EXP-001 (F1 0.673)
   - `docs/experiment_registry.md` — EXP-002 → COMPLETED
   - `configs/exp002_codebert.yaml` — `experiment.status`
   - Kiểm tra `run_metadata.json` có `device_actual: cuda:0` + `cuda_device_name`
     (chứng minh fix F-02 của DEC-017)
